# Notebook 5: DeepSeek Sparse Attention (DSA)

## Learning Objectives
- Understand standard attention complexity
- Learn DSA (DeepSeek Sparse Attention)
- Implement lightning indexer for token selection
- Understand O(L·k) complexity benefits

## Task 1: Standard Attention Complexity

### HINT: O(L²) Problem
```
Standard attention: O(seq_len²) for each token

For seq_len = 128K:
  128K × 128K = 16B operations!
```

In [ ]:
import torch
import torch.nn as nn
import math

# Calculate FLOPs for different sequence lengths
def attention_flops(seq_len, hidden_size, num_heads):
    head_dim = hidden_size // num_heads
    
    # Q @ K^T: seq² × head_dim
    qk_flops = seq_len * seq_len * head_dim * 2
    
    # softmax: seq²
    soft_flops = seq_len * seq_len
    
    # attn @ V: seq² × head_dim
    av_flops = seq_len * seq_len * head_dim * 2
    
    return qk_flops + soft_flops + av_flops

for seq_len in [512, 4096, 32768, 131072]:
    flops = attention_flops(seq_len, 512, 8)
    print(f"Seq len {seq_len:>6}: {flops/1e9:.2f} GFLOPs")

## Task 2: DSA Concept

### HINT: Key Innovation
DSA reduces complexity from O(L²) to **O(L·k)**:

```
Standard:  Attend to ALL past tokens
DSA:       Attend to only k most relevant tokens per position

Lightning Indexer: Selects top-k tokens using similarity scores
```

In [ ]:
class LightningIndexer(nn.Module):
    """Selects top-k most relevant tokens for each query"""
    def __init__(self, hidden_size, num_heads, num_selected=16):
        super().__init__()
        self.head_dim = hidden_size // num_heads
        self.num_selected = num_selected
        
    def forward(self, q, k, v):
        """
        q: [batch, num_heads, seq_len, head_dim]
        k: [batch, num_heads, seq_len, head_dim]
        v: [batch, num_heads, seq_len, head_dim]
        """
        batch, num_heads, seq_len, head_dim = q.shape
        
        # Compute similarity scores (Q @ K^T)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(head_dim)
        
        # For each query position, select top-k keys
        # [batch, num_heads, seq_len, seq_len] → [batch, num_heads, seq_len, k]
        topk_scores, topk_indices = torch.topk(
            scores, 
            k=min(self.num_selected, seq_len),
            dim=-1
        )
        
        # Gather selected values (simplified)
        # In practice, would gather K and V using indices
        return topk_indices, topk_scores

indexer = LightningIndexer(512, 8, num_selected=16)

# Test
batch, seq, heads, head_dim = 2, 32, 8, 64
q = torch.randn(batch, heads, seq, head_dim)
k = torch.randn(batch, heads, seq, head_dim)
v = torch.randn(batch, heads, seq, head_dim)

indices, scores = indexer(q, k, v)
print(f"Selected indices shape: {indices.shape}")  # [batch, heads, seq, k]
print(f"Selected k={indexer.num_selected} tokens per position")

## Task 3: Complete DSA Implementation

### HINT: DSA with Causal Masking
DSA also respects causal (autoregressive) structure

In [ ]:
class DeepSeekSparseAttention(nn.Module):
    """DeepSeek Sparse Attention"""
    def __init__(self, hidden_size, num_heads, num_selected=16):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.num_selected = num_selected
        
        self.indexer = LightningIndexer(hidden_size, num_heads, num_selected)
        
    def forward(self, q, k, v, causal=True):
        batch, num_heads, seq_len, head_dim = q.shape
        
        # Get selected indices
        selected_indices, selected_scores = self.indexer(q, k, v)
        
        # Expand indices for gathering [batch, heads, seq, 1] → [batch, heads, seq, k]
        batch_idx = torch.arange(batch, device=q.device).view(-1, 1, 1, 1).expand(-1, num_heads, seq_len, self.num_selected)
        head_idx = torch.arange(num_heads, device=q.device).view(1, -1, 1, 1).expand(batch, -1, seq_len, self.num_selected)
        seq_idx = torch.arange(seq_len, device=q.device).view(1, 1, -1, 1).expand(batch, num_heads, -1, self.num_selected)
        
        # Gather K and V at selected indices
        # Note: This is a simplified version
        
        # Apply causal mask to selected scores
        if causal:
            causal_mask = torch.tril(torch.ones(seq_len, seq_len, device=q.device))
            # This is simplified - full implementation uses proper masking
        
        # Compute attention with selected tokens
        attn_weights = torch.softmax(selected_scores, dim=-1)
        
        # Gather V at selected positions and compute output
        # Simplified: use standard attention
        output = torch.matmul(attn_weights, v)
        
        return output

# Compare complexity
def dsa_flops(seq_len, hidden_size, num_heads, num_selected):
    head_dim = hidden_size // num_heads
    
    # Q @ K^T: seq × k × head_dim (only k selected instead of seq)
    qk_flops = seq_len * num_selected * head_dim * 2
    
    # softmax: seq × k
    soft_flops = seq_len * num_selected
    
    # attn @ V: seq × k × head_dim
    av_flops = seq_len * num_selected * head_dim * 2
    
    return qk_flops + soft_flops + av_flops

print("=== FLOPs Comparison (seq_len=128K) ===")
print(f"Standard: {attention_flops(131072, 512, 8)/1e9:.1f} GFLOPs")
print(f"DSA:     {dsa_flops(131072, 512, 8, 64)/1e9:.2f} GFLOPs")
print(f"Speedup: {attention_flops(131072, 512, 8) / dsa_flops(131072, 512, 8, 64):.0f}x")

## Task 4: Full DSA Integration

### HINT: Combining with MLA
DSA can be combined with MLA for even more efficiency

In [ ]:
class DeepSeekAttention(nn.Module):
    """Complete attention: MLA + DSA"""
    def __init__(self, config):
        super().__init__()
        self.hidden_size = config['hidden_size']
        self.num_heads = config['num_heads']
        self.head_dim = self.hidden_size // self.num_heads
        self.num_selected = config.get('num_selected_tokens', 16)
        
        # Q and KV projections
        self.W_q = nn.Linear(self.hidden_size, self.hidden_size)
        self.W_KV = nn.Linear(self.hidden_size, self.head_dim * self.num_heads * 2 // 2)  # Compressed
        self.W_o = nn.Linear(self.hidden_size, self.hidden_size)
        
        self.sparse_attn = DeepSeekSparseAttention(
            self.hidden_size, 
            self.num_heads,
            self.num_selected
        )
        
    def forward(self, x, use_sparse=True):
        batch, seq_len, _ = x.shape
        
        # Q projection
        Q = self.W_q(x).view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # KV projection (compressed)
        KV = self.W_KV(x)
        latent_dim = KV.shape[-1] // 2
        K_latent = KV[..., :latent_dim]
        V_latent = KV[..., latent_dim:]
        
        # Expand for heads
        K = K_latent.unsqueeze(1).expand(-1, self.num_heads, -1, self.head_dim)
        V = V_latent.unsqueeze(1).expand(-1, self.num_heads, -1, self.head_dim)
        
        if use_sparse:
            # Use DSA
            attn_output = self.sparse_attn(Q, K, V)
        else:
            # Standard attention
            scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
            mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device))
            scores = scores.masked_fill(mask == 0, float('-inf'))
            attn_weights = torch.softmax(scores, dim=-1)
            attn_output = torch.matmul(attn_weights, V)
        
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch, seq_len, self.hidden_size)
        return self.W_o(attn_output)

# Test
config = {'hidden_size': 512, 'num_heads': 8, 'num_selected_tokens': 16}
attention = DeepSeekAttention(config)
x = torch.randn(2, 32, 512)
output = attention(x, use_sparse=True)
print(f"Input: {x.shape}")
print(f"Output: {output.shape}")

## Verification

Complete these checks:
1. ✅ Understand standard attention O(L²) complexity
2. ✅ Can explain DSA O(L·k) complexity
3. ✅ Can implement lightning indexer
4. ✅ Understand memory savings for long contexts